# 👥 Notebook 4: Customer Segmentation via Clustering
**Direct Mapping to Experiment 10:** Evaluating K-Means, Agglomerative/Hierarchical Clustering, and Gaussian Mixture Models (GMM) using Silhouette & Davies-Bouldin metrics.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.mixture import GaussianMixture
from sklearn.metrics import silhouette_score, davies_bouldin_score
import sys
sys.path.append('..')
from src.preprocessing import prepare_data


### 1. Elbow Method & Silhouette Scores for K-Means


In [ ]:
data = prepare_data('../data/online_shoppers_intention.csv')
X = data['X_train_scaled'].iloc[:3000]

inertias = []
silhouettes = []
k_range = range(2, 7)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X, labels))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))
ax1.plot(k_range, inertias, 'bo-')
ax1.set_title('Elbow Method (Inertia)')
ax1.set_xlabel('Number of Clusters (k)')

ax2.plot(k_range, silhouettes, 'go-')
ax2.set_title('Silhouette Score vs k')
ax2.set_xlabel('Number of Clusters (k)')
plt.show()


### 2. Multi-Algorithm Clustering Comparison


In [ ]:
# Benchmark K-Means vs Agglomerative vs GMM
k = 3
km = KMeans(n_clusters=k, random_state=42, n_init=10).fit_predict(X)
agg = AgglomerativeClustering(n_clusters=k).fit_predict(X)
gmm = GaussianMixture(n_components=k, random_state=42).fit_predict(X)

comparison = [
    {'Algorithm': 'K-Means', 'Silhouette': silhouette_score(X, km), 'Davies-Bouldin': davies_bouldin_score(X, km)},
    {'Algorithm': 'Agglomerative', 'Silhouette': silhouette_score(X, agg), 'Davies-Bouldin': davies_bouldin_score(X, agg)},
    {'Algorithm': 'Gaussian Mixture Model', 'Silhouette': silhouette_score(X, gmm), 'Davies-Bouldin': davies_bouldin_score(X, gmm)}
]
pd.DataFrame(comparison)


### 3. Cluster Profiling (Shopper Personas)


In [ ]:
# Profile clusters using original raw variables
df_profile = data['X_train'].iloc[:3000].copy()
df_profile['Cluster'] = km
df_profile['Revenue'] = data['y_train'].iloc[:3000]

profile_summary = df_profile.groupby('Cluster').agg({
    'ProductRelated_Duration': 'mean',
    'PageValues': 'mean',
    'BounceRates': 'mean',
    'Revenue': 'mean'
}).rename(columns={'Revenue': 'Conversion_Rate'})
print("Shopper Personas by Cluster:")
profile_summary
